# WeClone ROCm 单卡 LoRA 微调 Notebook

目标：在 AMD ROCm 环境里跑通 WeClone / LLaMA Factory 的单卡文本 LoRA 微调。

你的环境自检结果：

- GPU: AMD Radeon Graphics / Radeon PRO W7900D, gfx1100
- VRAM: 47.98 GB
- PyTorch ROCm 可用
- HIP: 7.2.x
- `torch.cuda.is_available() == True`

关键原则：不要直接执行 WeClone 官方的 `uv pip install --group main -e .`。它当前会安装 CUDA 版 torch，容易覆盖 ROCm 版 PyTorch。这个 notebook 会优先使用 Radeon Cloud 的 LLaMA Factory ROCm 镜像，只在缺包时补装。

Radeon Cloud 注意事项：免费额度按使用时长扣，哪怕没跑代码也会持续扣费。实验完成后请关闭/销毁实例。代码、配置、数据、LoRA adapter 和日志放到 `/network-workspace`；基础模型权重很大，不放持久化目录，默认放 `/workspace/model-cache` 临时盘。

## 0. 使用方式

从上到下运行即可完成：证书修复、WeClone 拉取、依赖检查/补装、模型目录检查、训练配置生成、数据准备和训练前检查。

开关说明：

- `RUN_MODEL_DOWNLOAD = True`：自动下载基础模型到临时盘 `/workspace/model-cache`。模型不持久化，实例销毁后需要重下。
- `RUN_MAKE_DATASET = True`：已有 WeClone CSV / Telegram 导出时，自动生成 SFT 数据。
- `USE_LOCAL_ROLE_DATA = True`：优先使用当前仓库里的 `role_data/qixia` 完整角色数据；不存在时使用 `role_data/qixia_sample` 样例数据。
- `CREATE_TINY_DATASET = True`：没有任何角色数据时才创建 6 条假数据，只用于冒烟测试。
- `RUN_TRAIN = True`：真正开始训练。默认关闭，避免 Run All 时误烧免费额度。

存储规则：

- `/network-workspace/weclone-rocm`：持久化目录，保存 WeClone 代码、配置、数据、LoRA adapter、日志。
- `/workspace/model-cache/weclone-rocm`：临时模型缓存，保存 Qwen2.5-7B-Instruct 基础模型，不占 20GB 持久化额度。
- pip 安装的库和运行环境：断开连接或 Destroy 后会清空，所以 notebook 保留了自动化重建步骤。

执行状态看 Jupyter 左侧 cell gutter：`[*]` 表示正在运行，`[数字]` 表示运行结束；绿色/红色背景由 Radeon Cloud 的 Jupyter 前端控制，不在输出区伪造。


In [ ]:
from pathlib import Path
import os
import sys
import json
import subprocess
import textwrap

def run(cmd, cwd=None, timeout=None):
    print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")
    return result

print("python:", sys.version)
print("cwd:", os.getcwd())

## 1. 参数区

先不要改太多。第一轮建议用 Qwen2.5-7B-Instruct 文本 LoRA 跑通。48GB 显存后续可以再把 `cutoff_len`、batch 或模型大小调大。

In [ ]:
# 持久化目录：保存代码、配置、数据、LoRA adapter、日志。
PERSIST_ROOT = Path('/network-workspace') if Path('/network-workspace').exists() else Path('/workspace/persist')
PERSIST_DIR = PERSIST_ROOT / 'weclone-rocm'
WECLONE_DIR = PERSIST_DIR / 'WeClone'

# 临时模型目录：基础模型很大，不放 /network-workspace，避免占 20GB 持久化额度。
MODEL_CACHE_ROOT = Path('/workspace/model-cache/weclone-rocm')
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
MODEL_DIR = MODEL_CACHE_ROOT / 'Qwen2.5-7B-Instruct'

# 持久化输出目录。
ADAPTER_OUTPUT_DIR = PERSIST_DIR / 'model_output_rocm_lora_qwen25_7b'
LOG_DIR = PERSIST_DIR / 'logs'

# 开关。默认自动准备环境和模型，但不自动开训。
RUN_INSTALL = True          # 检查并补装 WeClone 本体和缺失依赖；不会覆盖 ROCm torch。
RUN_MODEL_DOWNLOAD = True  # 模型下载到 /workspace/model-cache，不持久化。
RUN_MAKE_DATASET = False   # 有 WeClone CSV/Telegram 原始数据时改成 True。
USE_LOCAL_ROLE_DATA = True  # 优先复制 role_data/qixia；没有完整数据时使用 role_data/qixia_sample。
CREATE_TINY_DATASET = False # 没有任何角色数据时才改成 True 创建假数据冒烟测试。
RUN_TRAIN = False          # 确认配置和显存后再改成 True。

# 保守训练参数。48GB 显存足够跑 Qwen2.5-7B-Instruct LoRA；先用小上下文降低不确定性。
CUTOFF_LEN = 1024
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
NUM_TRAIN_EPOCHS = 1
LORA_RANK = 8
LORA_DROPOUT = 0.25

LANGUAGE = 'zh'
DEFAULT_SYSTEM = '请你扮演一名人类，不要说自己是人工智能。请自然、简洁地回复对方。'
REPO_ROOT = Path.cwd()
LOCAL_ROLE_DATA_DIR = REPO_ROOT / 'role_data' / 'qixia'
SAMPLE_ROLE_DATA_DIR = REPO_ROOT / 'role_data' / 'qixia_sample'

for path in [PERSIST_DIR, MODEL_CACHE_ROOT, ADAPTER_OUTPUT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('PERSIST_DIR:', PERSIST_DIR)
print('WECLONE_DIR:', WECLONE_DIR)
print('MODEL_DIR:', MODEL_DIR)
print('ADAPTER_OUTPUT_DIR:', ADAPTER_OUTPUT_DIR)
print('LOG_DIR:', LOG_DIR)


## 2. ROCm / PyTorch 再确认

这里必须看到 `torch.version.hip` 有值。

In [ ]:
run("rocm-smi --showproductname --showmeminfo vram --showuse || true")

import torch
print("torch:", torch.__version__)
print("hip:", getattr(torch.version, "hip", None))
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())

assert getattr(torch.version, "hip", None), "当前不是 ROCm/HIP 版 PyTorch，先不要继续。"
assert torch.cuda.is_available(), "PyTorch 没有识别到 AMD GPU。"

props = torch.cuda.get_device_properties(0)
print("device:", torch.cuda.get_device_name(0))
print("total_memory_gb:", round(props.total_memory / 1024**3, 2))
print("bf16 supported:", torch.cuda.is_bf16_supported())

## 3. 修复 Radeon Cloud GitHub/HuggingFace 代理证书

Radeon Cloud 可能会给 GitHub / Docker / HuggingFace 返回平台自签的代理证书。
如果 `git clone` / `curl https://github.com` 报 `self-signed certificate` 或 `server certificate verification failed`，运行下面这一格，把平台代理证书加入系统 CA。

这比 `GIT_SSL_NO_VERIFY=true` 更安全：它不会关闭 TLS 校验，只是信任当前平台代理证书。

In [ ]:
run("apt-get update && apt-get install -y ca-certificates git curl openssl", timeout=600)

proxy_cert = "/usr/local/share/ca-certificates/radeon-proxy-github-docker-hf.crt"
run(rf"""
openssl s_client -connect github.com:443 -servername github.com -showcerts </dev/null 2>/dev/null \
  | sed -n '/BEGIN CERTIFICATE/,/END CERTIFICATE/p' \
  > {proxy_cert}
test -s {proxy_cert}
openssl x509 -in {proxy_cert} -noout -subject -issuer -dates
update-ca-certificates
git config --global http.sslCAInfo /etc/ssl/certs/ca-certificates.crt
git ls-remote https://github.com/xming521/WeClone.git HEAD
""", timeout=300)


## 4. 拉取 WeClone

如果目录已经存在，会跳过 clone。

In [ ]:
if not WECLONE_DIR.exists():
    run(f"git clone --depth 1 https://github.com/xming521/WeClone.git {WECLONE_DIR}", timeout=300)
else:
    print("WeClone already exists:", WECLONE_DIR)
    run("git rev-parse --short HEAD && git status --short", cwd=WECLONE_DIR)

assert (WECLONE_DIR / "pyproject.toml").exists(), "WeClone 目录不完整。"


## 5. 安装/检查依赖，但保留 ROCm torch

这一步不会使用 WeClone 官方 `main` 依赖组，也不会主动降级 LLaMA Factory ROCm 镜像里的包。流程是：

1. 以 `--no-deps` 安装 WeClone 本体。
2. 检查关键 Python 包是否可 import。
3. 只补装缺失包，避免覆盖 ROCm torch。
4. 安装后再次断言 `torch.version.hip` 仍然存在。

In [ ]:
if RUN_INSTALL:
    run('python -m pip install -U pip setuptools wheel', timeout=300)
    run('python -m pip install -e . --no-deps', cwd=WECLONE_DIR, timeout=300)

    import importlib.util
    import shlex

    required_modules = {
        'pandas': 'pandas',
        'pyjson5': 'pyjson5',
        'omegaconf': 'omegaconf',
        'click': 'click',
        'tqdm': 'tqdm',
        'rich': 'rich',
        'pydantic': 'pydantic',
        'loguru': 'loguru',
        'langchain-core': 'langchain_core',
        'openai': 'openai',
        'presidio-analyzer[transformers]': 'presidio_analyzer',
        'presidio-anonymizer': 'presidio_anonymizer',
        'spacy': 'spacy',
        'modelscope': 'modelscope',
        'transformers': 'transformers',
        'accelerate': 'accelerate',
        'datasets': 'datasets',
        'peft': 'peft',
        'trl': 'trl',
        'llamafactory': 'llamafactory',
        'sentencepiece': 'sentencepiece',
        'safetensors': 'safetensors',
        'tensorboard': 'tensorboard',
        'jieba': 'jieba',
        'rouge-chinese': 'rouge_chinese',
        'fire': 'fire',
    }

    missing = [pkg for pkg, module in required_modules.items() if importlib.util.find_spec(module) is None]
    print('missing packages:', missing)
    if missing:
        install_args = ' '.join(shlex.quote(pkg) for pkg in missing)
        run('python -m pip install --upgrade-strategy only-if-needed ' + install_args, timeout=1200)

    # WeClone make-dataset 的中文隐私过滤需要 spaCy 中文模型。
    if LANGUAGE == 'zh':
        import importlib.util
        if importlib.util.find_spec('zh_core_web_sm') is None:
            run(
                'python -m pip install '
                'https://github.com/explosion/spacy-models/releases/download/zh_core_web_sm-3.8.0/zh_core_web_sm-3.8.0-py3-none-any.whl',
                timeout=600,
            )
    elif LANGUAGE == 'en':
        import importlib.util
        if importlib.util.find_spec('en_core_web_sm') is None:
            run('python -m spacy download en_core_web_sm', timeout=600)
else:
    print('skip dependency install/check')

import torch
print('torch after install:', torch.__version__)
print('hip after install:', getattr(torch.version, 'hip', None))
assert getattr(torch.version, 'hip', None), '安装依赖后 torch 不再是 ROCm 版，需要重装 ROCm PyTorch。'

for pkg in ['torch', 'transformers', 'llamafactory', 'accelerate', 'peft', 'trl', 'modelscope', 'weclone']:
    run(f"python -m pip show {pkg} | sed -n '1,8p' || true")

## 6. 下载模型到临时盘

基础模型默认下载到 `/workspace/model-cache/weclone-rocm/Qwen2.5-7B-Instruct`，不占 `/network-workspace` 20GB 持久化额度。实例销毁后模型需要重下，但 LoRA adapter 会保存在持久化目录。

In [ ]:
if RUN_MODEL_DOWNLOAD:
    MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    if MODEL_DIR.exists() and any(MODEL_DIR.iterdir()):
        print('model dir already populated:', MODEL_DIR)
    else:
        run(f'modelscope download --model {MODEL_ID} --local_dir {MODEL_DIR}', cwd=WECLONE_DIR, timeout=None)
else:
    print('skip model download')

if MODEL_DIR.exists():
    print('model dir exists:', MODEL_DIR)
    print('sample files:', [p.name for p in list(MODEL_DIR.iterdir())[:10]])
else:
    print('MODEL_DIR does not exist yet. Set RUN_MODEL_DOWNLOAD=True, or update MODEL_DIR to your local model path.')

run(f'du -sh {MODEL_CACHE_ROOT} 2>/dev/null || true')

## 7. 生成 ROCm 训练配置

相对官方模板的关键改动：

- `flash_attn` 改成 `disabled`。
- 用 `bf16`，不用 `fp16`。
- 不启用 QLoRA / bitsandbytes。
- batch 和 cutoff_len 先保守。
- 数据清洗关闭，避免依赖 vLLM。

In [ ]:
import pyjson5

template_path = WECLONE_DIR / "settings.template.jsonc"
config_path = WECLONE_DIR / "settings.rocm.jsonc"

cfg = pyjson5.loads(template_path.read_text(encoding="utf-8"))

cfg["common_args"].update({
    "model_name_or_path": str(MODEL_DIR),
    "adapter_name_or_path": str(ADAPTER_OUTPUT_DIR),
    "template": "qwen",
    "default_system": DEFAULT_SYSTEM,
    "media_dir": "dataset/media",
    "finetuning_type": "lora",
    "enable_thinking": False,
    "trust_remote_code": True,
})

cfg["cli_args"].update({
    "full_log": False,
    "log_level": "INFO",
})

cfg["make_dataset_args"].update({
    "platform": "chat",
    "language": LANGUAGE,
    "include_type": ["text"],
    "add_time": False,
    "add_relation": False,
    "combine_msg_max_length": CUTOFF_LEN,
    "messages_max_length": CUTOFF_LEN,
    "online_llm_clear": False,
})
cfg["make_dataset_args"]["clean_dataset"]["enable_clean"] = False
cfg["make_dataset_args"]["vision_api"]["enable"] = False

train_args = cfg["train_sft_args"]
train_args.update({
    "stage": "sft",
    "dataset": "chat-sft",
    "dataset_dir": "./dataset/res_csv/sft",
    "use_fast_tokenizer": True,
    "lora_target": "q_proj,v_proj",
    "lora_rank": LORA_RANK,
    "lora_dropout": LORA_DROPOUT,
    "weight_decay": 0.1,
    "overwrite_cache": True,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "lr_scheduler_type": "cosine",
    "cutoff_len": CUTOFF_LEN,
    "logging_steps": 1,
    "save_steps": 50,
    "output_dir": str(ADAPTER_OUTPUT_DIR),
    "logging_dir": str(LOG_DIR),
    "learning_rate": 1e-4,
    "warmup_ratio": 0.1,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "plot_loss": True,
    "fp16": False,
    "bf16": True,
    "flash_attn": "disabled",
    "gradient_checkpointing": True,
    "preprocessing_num_workers": 4,
    "dataloader_num_workers": 0,
    "do_train": True,
})
train_args.pop("deepspeed", None)
train_args["quantization"] = {}

config_text = json.dumps(cfg, ensure_ascii=False, indent=2)
config_path.write_text(config_text, encoding="utf-8")
print("wrote:", config_path)
print(config_text[:3000])

## 8A. 数据入口：WeClone 原始聊天数据

如果你有 WeClone 支持的 CSV 数据，把文件放成类似这样：

```text
WeClone/dataset/csv/person_a/chat_0_1000.csv
WeClone/dataset/csv/person_b/chat_0_1000.csv
```

CSV 至少需要这些列：

```text
id,MsgSvrID,type_name,is_sender,talker,room_name,msg,src,CreateTime
```

`is_sender=1` 表示你要克隆的那个人，`is_sender=0` 表示对方。确认数据放好后，把 `RUN_MAKE_DATASET=True` 再运行下一格。

In [ ]:
if RUN_MAKE_DATASET:
    run("find dataset/csv -maxdepth 3 -type f | head -20", cwd=WECLONE_DIR)
    run(f"weclone-cli --config-path {config_path} make-dataset", cwd=WECLONE_DIR, timeout=None)
else:
    print("skip make-dataset")

## 8B. 数据入口：直接使用 ShareGPT 训练数据

完整训练数据应放在当前仓库的 `role_data/qixia/`：

```text
role_data/qixia/sft-my.json
role_data/qixia/dataset_info.json
```

公开仓库只带 `role_data/qixia_sample/` 小样例。样例可以验证流程，但不适合正式训练。


In [ ]:
import shutil

sft_dir = WECLONE_DIR / "dataset" / "res_csv" / "sft"
sft_dir.mkdir(parents=True, exist_ok=True)

def copy_role_data(source_dir: Path, label: str) -> bool:
    source_sft = source_dir / "sft-my.json"
    source_info = source_dir / "dataset_info.json"
    if not source_sft.exists() or not source_info.exists():
        return False
    shutil.copy2(source_sft, sft_dir / "sft-my.json")
    shutil.copy2(source_info, sft_dir / "dataset_info.json")
    print(f"use {label} dataset:", source_dir)
    return True

used_dataset = False
if USE_LOCAL_ROLE_DATA:
    used_dataset = copy_role_data(LOCAL_ROLE_DATA_DIR, "full qixia")
    if not used_dataset:
        used_dataset = copy_role_data(SAMPLE_ROLE_DATA_DIR, "sample qixia")

if not used_dataset:
    dataset_info = {
        "chat-sft": {
            "file_name": "./sft-my.json",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "system": "system"},
            "tags": {
                "role_tag": "role",
                "content_tag": "content",
                "user_tag": "user",
                "assistant_tag": "assistant"
            }
        }
    }
    (sft_dir / "dataset_info.json").write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding="utf-8")

    tiny_data = [
        {"system": DEFAULT_SYSTEM, "messages": [{"role": "user", "content": "今天好累，感觉啥都不想做。"}, {"role": "assistant", "content": "那就先别硬撑了，休息一下，洗个热水澡或者早点睡。"}]},
        {"system": DEFAULT_SYSTEM, "messages": [{"role": "user", "content": "我刚刚把事情搞砸了。"}, {"role": "assistant", "content": "先别急着否定自己，看看还能补救哪一步，能补一点是一点。"}]},
        {"system": DEFAULT_SYSTEM, "messages": [{"role": "user", "content": "你在忙吗？"}, {"role": "assistant", "content": "还好，怎么啦？"}]},
        {"system": DEFAULT_SYSTEM, "messages": [{"role": "user", "content": "这个模型训练是不是很麻烦？"}, {"role": "assistant", "content": "第一次会有点绕，先把环境和数据跑通，后面就是调参数。"}]},
        {"system": DEFAULT_SYSTEM, "messages": [{"role": "user", "content": "晚上吃什么？"}, {"role": "assistant", "content": "想省事就点个清淡的，别太油。"}]},
        {"system": DEFAULT_SYSTEM, "messages": [{"role": "user", "content": "我有点焦虑。"}, {"role": "assistant", "content": "先把最急的一件事写下来，只处理它，别同时想太多。"}]}
    ]

    sft_file = sft_dir / "sft-my.json"
    if CREATE_TINY_DATASET:
        sft_file.write_text(json.dumps(tiny_data, ensure_ascii=False, indent=2), encoding="utf-8")
        print("wrote tiny dataset:", sft_file)
    else:
        raise FileNotFoundError(
            "No role dataset found. Put full data in role_data/qixia, "
            "or keep role_data/qixia_sample in the repository, or set CREATE_TINY_DATASET=True."
        )

sft_file = sft_dir / "sft-my.json"
data = json.loads(sft_file.read_text(encoding="utf-8"))
print("num examples:", len(data))
print(json.dumps(data[0], ensure_ascii=False, indent=2)[:1000])


## 9. 训练前检查

这里会检查配置、数据、模型目录、LoRA 输出目录和 GPU 显存。基础模型可以在临时盘；LoRA 输出必须在 `/network-workspace` 或 fallback 持久化目录下。

In [ ]:
print('config exists:', config_path.exists(), config_path)
print('dataset exists:', (sft_dir / 'sft-my.json').exists(), sft_dir / 'sft-my.json')
print('dataset_info exists:', (sft_dir / 'dataset_info.json').exists())
print('model exists:', MODEL_DIR.exists(), MODEL_DIR)
print('adapter output:', ADAPTER_OUTPUT_DIR)
print('log dir:', LOG_DIR)
run('rocm-smi --showmeminfo vram --showuse || true')
run(f'du -sh {PERSIST_DIR} 2>/dev/null || true')
run(f'du -sh {MODEL_CACHE_ROOT} 2>/dev/null || true')

if not MODEL_DIR.exists():
    raise FileNotFoundError(f'Model directory not found: {MODEL_DIR}. Set RUN_MODEL_DOWNLOAD=True or update MODEL_DIR.')
if not (sft_dir / 'sft-my.json').exists():
    missing_sft = sft_dir / 'sft-my.json'
    raise FileNotFoundError(f'SFT dataset not found: {missing_sft}')
print('RUN_TRAIN:', RUN_TRAIN)

## 10. 开始训练

确认模型和数据都存在、显存空出来后，把参数区的 `RUN_TRAIN = True` 再运行。

In [ ]:
if RUN_TRAIN:
    env = os.environ.copy()
    env['HIP_VISIBLE_DEVICES'] = '0'
    env['CUDA_VISIBLE_DEVICES'] = '0'
    env['PYTORCH_HIP_ALLOC_CONF'] = 'expandable_segments:True'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['WECLONE_CONFIG_PATH'] = str(config_path)

    train_log = LOG_DIR / 'train_sft.log'
    cmd = f'weclone-cli --config-path {config_path} train-sft'
    print(f'$ {cmd}')
    print('log:', train_log)
    with train_log.open('w', encoding='utf-8') as log_f:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=WECLONE_DIR,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_f.write(line)
        ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f'train failed with exit code {ret}; see log: {train_log}')
else:
    print('skip train. Set RUN_TRAIN=True after checking model/data/VRAM.')

## 11. 训练结果检查

训练成功后，LoRA adapter 会保存在持久化目录 `ADAPTER_OUTPUT_DIR`，默认是 `/network-workspace/weclone-rocm/model_output_rocm_lora_qwen25_7b`。


In [ ]:
output_dir = ADAPTER_OUTPUT_DIR
if output_dir.exists():
    run(f'find {output_dir} -maxdepth 2 -type f | sort | head -80')
    run(f'du -sh {output_dir} 2>/dev/null || true')
else:
    print('output dir not found yet:', output_dir)

## 12. 结束前检查

Radeon Cloud 按使用时长扣额度。训练结束、文件确认保存后，记得关闭/销毁实例。

持久化保存：

- WeClone 仓库和配置：`/network-workspace/weclone-rocm/WeClone`
- 数据集：`/network-workspace/weclone-rocm/WeClone/dataset/res_csv/sft/sft-my.json`
- LoRA adapter：`/network-workspace/weclone-rocm/model_output_rocm_lora_qwen25_7b`
- 日志：`/network-workspace/weclone-rocm/logs`

不持久化保存：

- 基础模型：`/workspace/model-cache/weclone-rocm/Qwen2.5-7B-Instruct`，实例销毁后需要重下。

In [ ]:
print('PERSIST_DIR:', PERSIST_DIR)
print('WeClone dir:', WECLONE_DIR, WECLONE_DIR.exists())
print('model dir (temporary):', MODEL_DIR, MODEL_DIR.exists())
print('dataset:', sft_dir / 'sft-my.json', (sft_dir / 'sft-my.json').exists())
print('adapter output:', ADAPTER_OUTPUT_DIR, ADAPTER_OUTPUT_DIR.exists())
print('log dir:', LOG_DIR, LOG_DIR.exists())
run(f'du -sh {PERSIST_DIR} 2>/dev/null || true')
run(f'du -sh {MODEL_CACHE_ROOT} 2>/dev/null || true')

## 13. 常见问题

- 如果 `torch.version.hip` 变成 `None`：说明安装依赖时 torch 被覆盖了，需要重装 ROCm PyTorch，不能继续训练。
- 如果 GitHub/HuggingFace 证书报 self-signed：先运行第 3 节，导入 Radeon Cloud 的平台代理证书。
- 如果模型目录不存在：打开 `RUN_MODEL_DOWNLOAD=True`，模型会下载到 `/workspace/model-cache` 临时盘。
- 如果 OOM：先确认没有别的进程占显存；再把 `CUTOFF_LEN=512`，`PER_DEVICE_TRAIN_BATCH_SIZE=1`，保持 `gradient_checkpointing=True`。
- 如果报 `flash_attn`：确认配置里的 `flash_attn` 是 `disabled`，不要安装 CUDA FlashAttention。
- 如果想跑 QLoRA：ROCm 下需要额外处理 bitsandbytes，第一版不建议。
- 训练结束后重点保存 LoRA adapter 和日志；基础模型不放持久化目录，避免占满 `/network-workspace`。